# Cancer🔬 Classification: Inference with ⚡`lightning`

**It is continuation of Training: https://www.kaggle.com/code/jirkaborovec/cancer-subtype-tiles-w-lightning-timm-models**

In [1]:
!ls /kaggle/input/pyvips-python-and-deb-package-gpu
# intall the deb packages
!yes | dpkg -i --force-depends /kaggle/input/pyvips-python-and-deb-package-gpu/linux_packages/archives/*.deb
# install the python wrapper
!pip install pyvips -f /kaggle/input/pyvips-python-and-deb-package-gpu/python_packages/ --no-index

linux_packages	python_packages
Selecting previously unselected package apparmor.
(Reading database ... 113818 files and directories currently installed.)
Preparing to unpack .../apparmor_3.0.4-2ubuntu2.2_amd64.deb ...
Unpacking apparmor (3.0.4-2ubuntu2.2) ...
Selecting previously unselected package autoconf.
Preparing to unpack .../autoconf_2.71-2_all.deb ...
Unpacking autoconf (2.71-2) ...
Selecting previously unselected package automake.
Preparing to unpack .../automake_13a1.16.5-1.3_all.deb ...
Unpacking automake (1:1.16.5-1.3) ...
Selecting previously unselected package autotools-dev.
Preparing to unpack .../autotools-dev_20220109.1_all.deb ...
Unpacking autotools-dev (20220109.1) ...
Selecting previously unselected package bzip2-doc.
Preparing to unpack .../bzip2-doc_1.0.8-5build1_all.deb ...
Unpacking bzip2-doc (1.0.8-5build1) ...
Selecting previously unselected package file.
Preparing to unpack .../file_13a5.41-3ubuntu0.1_amd64.deb ...
Unpacking file (1:5.41-3ubuntu0.1) ...
Sele

In [2]:
import os, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATASET_FOLDER = "/kaggle/input/UBC-OCEAN/"
IMAGES_FOLDER = "./test_tiles"
BAG_SIZE = 6
MAX_SAMPLE_PER_IMAGE = 50 #BAG_SIZE*5
INPUT_SIZE = 512
TILE_SIZE = 1024
BATCH_SIZE = 2
NUM_WORKERS = 2
os.environ['VIPS_CONCURRENCY'] = '4'
os.environ['VIPS_DISC_THRESHOLD'] = '15gb'

## Data for inference

**Note, we canot extract all tiles from all imges because of unsufficient space/storage**

This needs porting several code (classes) form the training notebook:

- extracting the tiles from whole image
- validation argmention, mainly color mean & STD

In [3]:
df_train = pd.read_csv(os.path.join(DATASET_FOLDER, "train.csv"))
LABELS = ['CC', 'EC', 'HGSC', 'LGSC','MC','Stroma','Necrosis']
print(f"{LABELS=}")

LABELS=['CC', 'EC', 'HGSC', 'LGSC', 'MC', 'Stroma', 'Necrosis']


In [4]:
df_train.head()

,image_id,label,image_width,image_height,is_tma
0,4,HGSC,23785,20008,False
1,66,LGSC,48871,48195,False
2,91,HGSC,3388,3388,True
3,281,LGSC,42309,15545,False
4,286,EC,37204,30020,False


In [5]:
df_train.groupby('image_id').get_group(4)['is_tma'].item()

False

In [6]:
TH_TMA_FILE_SIZE   = 1.5
!du -s -m /kaggle/input/UBC-OCEAN/test_images/* > amount.csv
df_size = pd.read_csv("amount.csv", delimiter='\t', header=None).rename(columns={0:"size_mb", 1:"file"})
df_size["image_id"] = df_size["file"].apply(lambda x: int(os.path.basename(x).split(".")[0] ) )
df_size["size_mb_log10"] = np.log10(df_size["size_mb"].values)
df_size["is_tma"] = df_size["size_mb_log10"] < TH_TMA_FILE_SIZE
# df_size.set_index('image_id',inplace = True)
df_size

,size_mb,file,image_id,size_mb_log10,is_tma
0,621,/kaggle/input/UBC-OCEAN/test_images/41.png,41,2.793092,False


In [7]:
import os
import pyvips
import numpy as np
import random
from PIL import Image
import gc

def drop_bg(tile, area_thresh = 0.6, white_thresh = 220):
    drop_black = False
    drop_white = False
    mask_bg = np.sum(tile, axis=2) == 0
    if np.sum(mask_bg) >= (np.prod(mask_bg.shape) * area_thresh): # too much black
        drop_black = True
    tile[mask_bg, :] = 255
    mask_bg = np.mean(tile, axis=2) > white_thresh
    if np.sum(mask_bg) >= (np.prod(mask_bg.shape) * area_thresh):# too much white
        drop_white = True
#     if drop_black:
#         print('too many blacks')
#     if drop_white:
#         print('too many whites')
    return drop_black or drop_white

def evaluate_tumor_quality(model, dataloader):
    model.eval()
    model = model.to('cuda')
    preds = []
    for batch in dataloader:
        pred = model(batch.cuda())
        preds.append(pred.detach().cpu())
        
    
    preds = torch.cat(preds)
#     print(preds.shape)
    preds = torch.softmax(preds, axis = 1)
    labels = torch.argmax(preds, axis = 1).numpy()
    proba = torch.max(preds, axis=1).values.numpy()
#     print(labels, proba)
    has_tumor = [l in range(5) for l, p in zip(labels, proba)]
#     print(list(zip(labels, proba, has_tumor)))
    model = model.to('cpu')
    return has_tumor

def extract_image_tiles(
    img_wsi,
    tile_size = 1024,
    resize_shape = None,
    max_sample=None,
    check_drop_bg  =True,
    patch_classifier_model = None
):
    im = img_wsi# pyvips.Image.new_from_file(p_img)
    w = h = tile_size
    # https://stackoverflow.com/a/47581978/4521646
    idxs = [(y, y + h, x, x + w) for y in range(0, im.height, h) for x in range(0, im.width, w)]
    print('total patches', len(idxs))
    
    np.random.seed(42)
    if max_sample:
        np.random.shuffle(idxs)
        idxs = idxs[:max_sample]
    
    dataset_bg_det = BG_Det_Dataset(im, idxs, tile_size = tile_size)
    dataloader_bg_det = DataLoader(
            dataset_bg_det, batch_size=4, num_workers=0, shuffle=False,
        )
    del im
    for _ in range(5):
        gc.collect()
    bool_indices = []
    tiles = []
    for tiles_batch, batch_bool in tqdm(dataloader_bg_det):
        bool_indices+=list(batch_bool)
        tiles.append(tiles_batch)
    tiles = np.concatenate(tiles)
    tile_list_no_bg = np.array(tiles)[~np.array(bool_indices)]
    print('patches after filtering background', len(tile_list_no_bg))
    
    if len(tile_list_no_bg) == 0:
        return tiles[:BAG_SIZE*4]

    dataset = PatchesDataset(
        [Image.fromarray(tile) for tile in tile_list_no_bg], 
        patch_classifier_model['preprocess']
    )
    dataloader = DataLoader(
            dataset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, shuffle=False,
        )
    has_tumor = evaluate_tumor_quality(patch_classifier_model['model'], dataloader)
    
    tile_list_no_bg_pc = np.array(tile_list_no_bg)[has_tumor]
    print('patches after filtering background and patch classification', len(tile_list_no_bg_pc))

    if len(tile_list_no_bg_pc) < BAG_SIZE:
        print('patches got removed due to bg removal and patch classifier, adding random bags')
#         tile_list_no_bg_pc = list(tile_list_no_bg_pc)
#         tile_list_no_bg_pc += list(tile_list_no_bg[:BAG_SIZE*4])
#         print(np.array(tile_list_no_bg_pc).shape)
        
    
    del dataset, dataloader
    for _ in range(5):
        gc.collect()
#     for img in tile_list_no_bg_pc[:10]:
#         plt.imshow(img)
#         plt.show()
    return tile_list_no_bg_pc

In [8]:
from torchvision import transforms as T

img_color_mean = [0.8721593659261734, 0.7799686061900686, 0.8644588534918227]
img_color_std = [0.08258995918115268, 0.10991684444009092, 0.06839816226731532]

VALID_TRANSFORM = T.Compose([
    T.Resize((INPUT_SIZE,INPUT_SIZE)),
    T.ToTensor(),
    #T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    T.Normalize(img_color_mean, img_color_std),  # custom
])

In [9]:
import torch
from PIL import Image
from torch.utils.data import Dataset

class TilesDataset(Dataset):

    def __init__(
        self,
        tiles,
        bag_size,
        transforms
    ):
        self.transforms = transforms
        self.imgs = tiles
        self.available_bags = len(self.imgs)//bag_size if len(self.imgs)//bag_size>0 else 1
        self.bag_size = bag_size
        self.imgs_bags = [self.imgs[i*bag_size:i*bag_size+bag_size] for i in range(self.available_bags) ]

    def __getitem__(self, idx: int) -> tuple:
        bag = self.imgs_bags[idx]
        # augmentation
        if self.transforms:
            bag =[self.transforms(Image.fromarray(item))  for item in bag]
        #print(f"img dim: {img.shape}")
#         print(bag)
        return torch.stack(bag)#, torch.tensor(labels).to(int)


    def __len__(self) -> int:
        return self.available_bags
    
class PatchesDataset(Dataset):
    def __init__(
        self,
        tiles,
        transforms
    ):
        self.transforms = transforms
        self.tiles = tiles

    def __getitem__(self, idx: int) -> tuple:
        tile = self.tiles[idx]
        tile = self.transforms(tile)
        return tile

    def __len__(self) -> int:
        return len(self.tiles)
    
class BG_Det_Dataset(Dataset):
    def __init__(
        self,
        im,
        idxs,
        tile_size
    ):
        self.im = im
        self.idxs = idxs
        self.tile_size = tile_size

    def __getitem__(self, idx: int) -> tuple:
        y, y_, x, x_ = self.idxs[idx]
        # make tile
        w = h = self.tile_size
        tile = self.im.crop(x, y, min(w, self.im.width - x), min(h, self.im.height - y)).numpy()[..., :3]
        if tile.shape[:2] != (h, w):
            tile_ = tile
            tile_size = (h, w) if tile.ndim == 2 else (h, w, tile.shape[2])
            tile = np.zeros(tile_size, dtype=tile.dtype)
            tile[:tile_.shape[0], :tile_.shape[1], ...] = tile_
        drop_bool = drop_bg(tile.copy())
#         if drop_bool == True:
#             tile = []
        return tile, drop_bool
    def __len__(self) -> int:
        return len(self.idxs)

## CNN Model

We start with some stanrd CNN models taken from torch vision.

In [10]:
import timm
import torch
import torchvision
import pytorch_lightning as pl
from torch import nn
from torch.nn import functional as F


enc= timm.create_model(
    'tiny_vit_21m_512.dist_in22k_ft_in1k', pretrained=False, num_classes=7)
class Model(nn.Module):
    def __init__(self,enc):
        super().__init__()
        self.enc = enc
    def forward(self, x):
        return self.enc(x)
model_pc = Model(enc)
model_data = torch.load('/kaggle/input/2023-12-09-17-07-51/tiny_vit_21m_512.dist_in22k_ft_in1k/version_0/checkpoints/epoch=4-step=9925.ckpt')
model_pc.load_state_dict(model_data['state_dict'])

patch_classifier_model = {
    'model': model_pc,
    'preprocess': VALID_TRANSFORM
}

/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [11]:
import gc
gc.collect()

47

In [12]:
import torch
import torch.nn as nn

class AdaptiveConcatPool2d(torch.nn.Module):
    "Layer that concats `AdaptiveAvgPool2d` and `AdaptiveMaxPool2d`"
    def __init__(self, size=None):
        super().__init__()
        self.size = size or 1
        self.ap = torch.nn.AdaptiveAvgPool2d(self.size)
        self.mp = torch.nn.AdaptiveMaxPool2d(self.size)
    def forward(self, x): return torch.cat([self.mp(x), self.ap(x)], 1)


class Model_BOW(nn.Module):
    def __init__(self, enc, feature_dim,
                 num_class = 5):
        super().__init__()
    #         self.conv_bag = torch.nn.Conv2d(in_channels=12*3,out_channels=3,kernel_size=3,padding=1)
    #         self.net = net
    #         self.arch = net.pretrained_cfg.get('architecture')
        self.num_classes = num_class
        self.enc = enc
        self.feature_dim = feature_dim
        self.head = nn.Sequential(
            AdaptiveConcatPool2d(),
            torch.nn.Flatten(),
            nn.Linear(2*feature_dim,512),
            torch.nn.ReLU(),
            #torch.nn.Mish(),
            torch.nn.LayerNorm(512),
            nn.Linear(512,256),
            torch.nn.ReLU(),
            torch.nn.Mish(),
            torch.nn.LayerNorm(256),
            #torch.nn.Dropout(0.5),
            torch.nn.Linear(256,self.num_classes)
        )


        
    def forward(self, x, x_large):
        batch,bag,c,h,w = x.shape

        x = x.view(batch*bag, c,h,w)
        #x: bs*N x C x 4 x 4
        x = self.enc(x)
        _,c_out,h_out,w_out = x.shape
        #concatenate the output for tiles into a single map
        x = x.view(batch,bag,c_out,h_out,w_out)
        #x: bsxN x C x 4 x 4
        #print(x.shape)
        x_l = self.enc(x_large)
        #print(x_l.shape)
        x_l = x_l.unsqueeze(1)
        x=torch.cat([x,x_l], axis=1)

        #x_l = x_l.repeat(1,bag,1,1,1)
        #x = x+x_l # add?

        x = x.permute(0,2,1,3,4).contiguous() # bs,C,N,H,W
        #print(x.shape)
        x = x.view(batch,c_out,h_out*(bag+1),w_out)
        #print(x.shape)
        x = self.head(x)

        #x: bs x n
        return x

net= timm.create_model(
    'maxvit_tiny_tf_512', pretrained=False, num_classes=5)
net = nn.Sequential(*list(net.children())[:-1])
model_bow=Model_BOW(enc=net, feature_dim=512)
model_data = torch.load('/kaggle/input/2023-12-14-20-32-38/fold_0/maxvit_tiny_tf_512/version_0/checkpoints/epoch=99-step=3400.ckpt')
model_bow.load_state_dict(model_data['state_dict'])

<All keys matched successfully>

## Test data & submission

lest load sample submission and add append images we can predict

In [13]:
df_test = pd.read_csv(os.path.join(DATASET_FOLDER, "test.csv"))
# default label
df_test['label'] = ['HGSC'] * len(df_test)
# labels = list(df_train["label"].unique())
print(f"Dataset/test size: {len(df_test)}")
display(df_test.head())

Dataset/test size: 1


,image_id,image_width,image_height,label
0,41,28469,16987,HGSC


In [14]:
!cat /kaggle/input/UBC-OCEAN/sample_submission.csv

image_id,label
41,HGSC


## Inference

In [15]:
def infer_single_image(model, patch_classifier_model, row, train):
    row = dict(row)
    # prepare data - cut and load tiles
    if train:
        split = 'train'
    else:
        split = 'test'
    path = os.path.join(DATASET_FOLDER, "{}_images".format(split), f"{str(row['image_id'])}.png")
    img_wsi = pyvips.Image.new_from_file(path)

    tiles = extract_image_tiles(
        img_wsi,
        tile_size=TILE_SIZE, 
        max_sample=MAX_SAMPLE_PER_IMAGE,
        patch_classifier_model = patch_classifier_model
    )
#     print(tiles[0].shape)

    del img_wsi
    for _ in range(5):
        gc.collect()
    
    if len(tiles)==0:
        row['label'] = 'Other'
    else:
        
        img_global = Image.open(os.path.join(
            '/kaggle/input/UBC-OCEAN/{}_thumbnails'.format(split),
            '{}_thumbnail.png'.format(row['image_id'])
        ))
    #     img_global = np.array(img_global)
        img_global = data_transforms_valid(image = np.array(img_global))['image']
        img_global = img_global.unsqueeze(0)


        dataset = TilesDataset(tiles, bag_size=BAG_SIZE,transforms=VALID_TRANSFORM)
        dataloader = DataLoader(
                dataset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, shuffle=False,
        #                            multiprocessing_context=get_context('loky')
            )
        print('bags', len(dataset))
        # iterate over images and collect predictions
        preds = []
        model = model.to('cuda')
        for imgs in dataloader:
            #print(f"{imgs.shape}")
            img_global_batch = torch.cat([img_global.clone()]*len(imgs))
            img_global_batch = img_global_batch.cuda()
            with torch.no_grad():
                pred = model(imgs.cuda(), img_global_batch)
            preds.append(pred.detach().cpu())
        model = model.to('cpu')
        # decide label
        preds = torch.cat(preds)
        print(preds.shape)
        preds = torch.mean(preds, axis=0)
        preds = torch.softmax(preds.view(-1), axis=0)
        print(preds)
        lb = np.argmax(preds)
        row['label'] = LABELS[lb]

        del dataset, dataloader
        for _ in range(5):
            gc.collect()

    return row


In [16]:
def infer_single_image_tma(model, row, train):
    
    model = model.to('cuda')
    model.eval()
    row = dict(row)
    if train:
        split = 'train'
    else: 
        split = 'test'
    path = os.path.join('/kaggle/input/UBC-OCEAN/{}_images/{}.png'.format(split, row['image_id']))
    img = Image.open(path)
    img = np.array(img)
    pred = np.zeros((1,7))
    for tx in rotate_tx:
        img_tx = tx(image = img)['image']   
        img_tx = data_transforms_valid(image = img_tx)['image']
        img_tx = img_tx.unsqueeze(0)
        #print(f"{imgs.shape}")
        with torch.no_grad():
#             print(model(img_tx.cuda()).detach().cpu().numpy())
            pred += model(img_tx.cuda()).detach().cpu().numpy()
    # decide label
    model = model.to('cpu')
    lb = np.argmax(pred)
    row['label'] = LABELS[lb]
    return row

In [17]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

item = (2048, 512)
data_transforms_valid = A.Compose([
            A.PadIfNeeded(item[0], item[0]),
            A.CenterCrop(item[0], item[0]),
            A.Resize(item[1], item[1]),
            A.Normalize(
                mean = img_color_mean, #[0.485, 0.456, 0.406], 
                std = img_color_std, #[0.229, 0.224, 0.225], 
                max_pixel_value=255.0, 
                p=1.0
            ),
            ToTensorV2()], p=1.)

tx90 = A.Compose(
    [
        A.Rotate(limit=(90,90),always_apply=True, p=1.0),
#         A.Resize(4,4)
    ], p=1.0)
tx180 = A.Compose(
    [
        A.Rotate(limit=(90,90),always_apply=True, p=1.0),
#         A.Resize(4,4)
    ]*2, p=1.0)
tx270 = A.Compose(
    [
        A.Rotate(limit=(90,90),always_apply=True, p=1.0),
#         A.Resize(4,4)
    ]*3, p=1.0)
rotate_tx = [tx90, tx180, tx270]

In [18]:
import shutil
from torch.utils.data import DataLoader
from joblib.externals.loky.backend.context import get_context
from tqdm.auto import tqdm 
import time

model_bow.eval()
TRAIN = False

submission = []
if TRAIN:
    df_select = df_train.copy()
else:
    df_select = df_test.copy()
    
# df_select = df_select[50:100]    

for i, row in tqdm(df_select.iterrows(), total = len(df_select)):
    print(row.to_list())
    if TRAIN:
        is_tma = df_select.groupby('image_id').get_group(row['image_id'])['is_tma'].item()
    else:
        is_tma = df_size.groupby('image_id').get_group(row['image_id'])['is_tma'].item()
    if is_tma:
        row = infer_single_image(model_bow, patch_classifier_model, row, TRAIN)

        row = infer_single_image_tma(model_pc, row, TRAIN)
        if row['label'] in ['Stroma', 'Necrosis']:
            row['label'] = 'Other'
        submission.append(dict(row))
    else:
        start = time.time()
        row = infer_single_image(model_bow, patch_classifier_model, row, TRAIN)
#         print(row)
        submission.append(dict(row))
        print(time.time()-start)
    print(row)
df_sub = pd.DataFrame(submission)


  0%|          | 0/1 [00:00<?, ?it/s]

[41, 28469, 16987, 'HGSC']
total patches 476


  0%|          | 0/13 [00:00<?, ?it/s]

patches after filtering background 22
patches after filtering background and patch classification 0
patches got removed due to bg removal and patch classifier, adding random bags
30.581491947174072
{'image_id': 41, 'image_width': 28469, 'image_height': 16987, 'label': 'Other'}


In [19]:
# from sklearn import metrics

# print(metrics.balanced_accuracy_score(df_train[~df_train['is_tma']].label.values, df_sub[~df_train['is_tma']].label.values))

# print(metrics.confusion_matrix(df_train[~df_train['is_tma']].label.values, df_sub[~df_train['is_tma']].label.values))

## Finalize - export submission

In [20]:
display(df_sub.head())
df_sub[["image_id", "label"]].to_csv("submission.csv", index=False)

! head submission.csv

,image_id,image_width,image_height,label
0,41,28469,16987,Other


image_id,label
41,Other
